# Bloch vs PINN (Pointwise Multi-Band Picking Only)

This notebook compares Bloch analytical branches against PINN **pointwise** picking with selectable band count (`n_bands = 2` or `3`).

No continuity ridge tracking is used.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path('.').resolve()))

from pinn_dispersion_from_mat import (
    load_pinn_results,
    resample_to_uniform,
    compute_spectrum,
    compute_dos,
    detect_band_gaps,
    extract_ridge_in_band,
    linear_dispersion,
)
from dispersion_theory import monoatomic_dispersion, mass_in_mass_dispersion

In [ ]:
# ---------------- USER SETTINGS ----------------
mat_file = '../Result/pinn_data.mat'

# Bloch reference parameters
k_coupling = 1.0
mx = 1.0
my = 0.3
k_int = 8.6483

# Pointwise multi-band extraction settings
n_bands = 3      # set to 2 or 3
omega_min = 0.01
skip_transient = 0.20
n_harmonic = 4
pts_per_cycle = 12
dip_threshold_dB = 5.0
min_distance = 5

In [ ]:
def pointwise_band_windows(omega_pos, spectrum, n_bands=3, omega_min=0.01,
                          dip_threshold_dB=5.0, min_distance=5):
    """Build frequency windows from DOS peaks (for pointwise ridge-in-band picking)."""
    i0 = np.searchsorted(omega_pos, omega_min)
    omega_clip = omega_pos[i0:]
    spec_clip = spectrum[:, i0:]

    dos, dos_dB = compute_dos(spec_clip)
    gap_freqs, band_freqs = detect_band_gaps(
        omega_clip, dos_dB,
        dip_threshold_dB=dip_threshold_dB,
        min_distance=min_distance,
    )

    if len(band_freqs) == 0:
        edges = np.linspace(omega_clip[0], omega_clip[-1], n_bands + 1)
        windows = [(float(edges[i]), float(edges[i + 1])) for i in range(n_bands)]
        return windows, omega_clip

    # keep top-N energetic passband peaks
    idx = np.searchsorted(omega_clip, band_freqs)
    idx = np.clip(idx, 0, len(dos)-1)
    keep = np.argsort(dos[idx])[::-1][:n_bands]
    peaks = np.sort(band_freqs[keep])

    # midpoint windows around selected peaks
    mids = 0.5 * (peaks[:-1] + peaks[1:]) if len(peaks) > 1 else np.array([])
    windows = []
    for i, w0 in enumerate(peaks):
        lo = omega_clip[0] if i == 0 else mids[i-1]
        hi = omega_clip[-1] if i == len(peaks)-1 else mids[i]

        left_gaps = gap_freqs[gap_freqs < w0]
        right_gaps = gap_freqs[gap_freqs > w0]
        if len(left_gaps) > 0:
            lo = max(lo, left_gaps[-1])
        if len(right_gaps) > 0:
            hi = min(hi, right_gaps[0])

        if hi <= lo:
            bw = max((omega_clip[-1] - omega_clip[0]) / (4 * max(1, len(peaks))), 1e-6)
            lo = max(omega_clip[0], w0 - bw)
            hi = min(omega_clip[-1], w0 + bw)

        windows.append((float(lo), float(hi)))

    return windows, omega_clip

In [ ]:
# PINN spectrum + pointwise multi-band ridges
t_raw, x_raw, params = load_pinn_results(mat_file)
X_raw = x_raw.T

omega_max_lin = linear_dispersion(np.pi, k_coupling, mx)
t_u, X_u, dt, omega_nyq = resample_to_uniform(
    t_raw, X_raw, omega_max_lin,
    n_harmonic=n_harmonic,
    pts_per_cycle=pts_per_cycle,
)
k_pos, omega_pos, spectrum = compute_spectrum(t_u, X_u, skip_transient=skip_transient)

windows, omega_clip = pointwise_band_windows(
    omega_pos, spectrum,
    n_bands=n_bands,
    omega_min=omega_min,
    dip_threshold_dB=dip_threshold_dB,
    min_distance=min_distance,
)

pointwise_ridges = []
for (w_lo, w_hi) in windows:
    i_lo = np.searchsorted(omega_pos, w_lo)
    i_hi = np.searchsorted(omega_pos, w_hi)
    omega_band = omega_pos[i_lo:i_hi]
    spec_band = spectrum[:, i_lo:i_hi]

    if len(omega_band) < 2:
        pointwise_ridges.append(np.full(len(k_pos), np.nan))
    else:
        pointwise_ridges.append(extract_ridge_in_band(k_pos, omega_band, spec_band, omega_min=omega_band[0]))

# Bloch curves
k_line = np.linspace(0, np.pi, 400)
omega_bloch_mono = monoatomic_dispersion(k_line, k_coupling, mx)
omega_bloch_lo, omega_bloch_hi = mass_in_mass_dispersion(k_line, k_coupling, mx, my, k_int)

print('Loaded:', mat_file)
print('Windows:')
for i, w in enumerate(windows, start=1):
    print(f'  Band {i}: [{w[0]:.4f}, {w[1]:.4f}]')

In [ ]:
# Comparison plot: Bloch vs pointwise multi-band
plt.figure(figsize=(10, 6.5))

# Bloch references
plt.plot(k_line/np.pi, omega_bloch_mono, 'k:', lw=1.8, label='Bloch monoatomic')
plt.plot(k_line/np.pi, omega_bloch_lo, 'k--', lw=2.0, label='Bloch acoustic (mass-in-mass)')
plt.plot(k_line/np.pi, omega_bloch_hi, 'k-.', lw=2.0, label='Bloch optical (mass-in-mass)')

for i, om in enumerate(pointwise_ridges, start=1):
    valid = np.isfinite(om)
    plt.plot(k_pos[valid]/np.pi, om[valid], lw=2.3, label=f'PINN pointwise band {i}')

plt.xlabel(r'Wavenumber $k/\pi$')
plt.ylabel(r'Frequency $\omega$ (rad/s)')
plt.title(f'Bloch vs PINN pointwise picking ({n_bands} bands)')
plt.grid(alpha=0.25)
plt.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()